In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.family'] = ['Liberation Sans', 'DejaVu Sans']

def gaussian_pdf(x, mu, sigma):
    """正态分布的概率密度函数：mu 是均值，sigma 是标准差"""
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

x = np.linspace(-6, 8, 500)  # 横轴：从 -6 到 8 取 500 个点

fig, ax = plt.subplots(figsize=(8, 4.5))

# 三条曲线：只有 mu 和 sigma 不同
ax.plot(x, gaussian_pdf(x, mu=0, sigma=1), label=r'$\mu=0,\ \sigma=1$')
ax.plot(x, gaussian_pdf(x, mu=2, sigma=1), label=r'$\mu=2,\ \sigma=1$')
ax.plot(x, gaussian_pdf(x, mu=2, sigma=0.5), label=r'$\mu=2,\ \sigma=0.5$')

ax.set_xlabel('x')
ax.set_ylabel('probability density')
ax.set_title('Normal distributions: effect of mu and sigma')
ax.legend()
plt.tight_layout()
plt.savefig('/mnt/results/gaussian_mu_sigma_demo.png', dpi=150)
plt.show()

In [2]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

rng = np.random.normal  # 简写

n = 300
X = np.random.normal(0, 1, n)          # 第1步：生成 X ~ N(0, 1)，300 个样本

# 第2步：加噪声 —— 就是把每个样本加上一个从高斯分布抽出来的随机数
noise_small = np.random.normal(0, 0.1, n)   # 噪声标准差 0.1
noise_big   = np.random.normal(0, 10,  n)   # 噪声标准差 10

Y_small = X + noise_small
Y_big   = X + noise_big

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

axes[0].scatter(X, Y_small, s=12, alpha=0.6, color='#0279EE')
axes[0].set_title('noise std = 0.1')
axes[0].set_xlabel('X (observed)')
axes[0].set_ylabel('Y = X + noise')

axes[1].scatter(X, Y_big, s=12, alpha=0.6, color='#FF9400')
axes[1].set_title('noise std = 10')
axes[1].set_xlabel('X (observed)')

plt.tight_layout()
plt.savefig('/mnt/results/noise_demo.png', dpi=150)
plt.show()

print(f"小噪声: X=2 附近时 Y 的标准差 ≈ {Y_small[np.abs(X-2)<0.2].std():.2f}")
print(f"大噪声: X=2 附近时 Y 的标准差 ≈ {Y_big[np.abs(X-2)<0.2].std():.2f}")

小噪声: X=2 附近时 Y 的标准差 ≈ 0.05
大噪声: X=2 附近时 Y 的标准差 ≈ 1.51


In [3]:
# ============================================================
# 第 1 步：生成模拟酶动力学数据
# 机制：Michaelis-Menten 方程  v = Vmax * S / (Km + S)
#   v    = 反应速率
#   S    = 底物浓度
#   Vmax = 最大速率（酶全部饱和时的速率）
#   Km   = 达到 Vmax 一半时的底物浓度
# ============================================================
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.family'] = ['Liberation Sans', 'DejaVu Sans']

# 真实参数（模拟“大自然”，实验员是不知道的）
VMAX_TRUE = 12.0   # umol/min
KM_TRUE   = 0.4    # mM

def michaelis_menten(S, vmax=VMAX_TRUE, km=KM_TRUE):
    return vmax * S / (km + S)

# 8 个实验测量点：浓度取对数间隔，覆盖 0.05 ~ 2.0 mM
# （实验员通常在对数尺度上均匀取样，低浓度区更密集）
rng = np.random.default_rng(seed=42)          # 固定种子：结果可复现
S_train = np.geomspace(0.05, 2.0, 8)          # 8 个底物浓度

# 测量 = 真实值 + 噪声（噪声标准差取真实速率的 5%）
noise_std = 0.05
v_train = michaelis_menten(S_train) + rng.normal(0, noise_std * VMAX_TRUE, size=8)

# 画出来看看
S_fine = np.linspace(0.01, 4.0, 400)          # 细密网格：画真实曲线用
v_true = michaelis_menten(S_fine)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(S_fine, v_true, 'k--', lw=1, label='true curve (unknown to experimenter)')
ax.scatter(S_train, v_train, s=60, color='#0279EE', zorder=3, label='8 measurements')
ax.set_xlabel('substrate concentration S (mM)')
ax.set_ylabel('reaction rate v (umol/min)')
ax.set_title('Simulated enzyme kinetics data')
ax.legend()
plt.tight_layout()
plt.savefig('/mnt/results/project_step1_data.png', dpi=150)
plt.show()

print('训练数据（实验员唯一能看到的东西）：')
for s, v in zip(S_train, v_train):
    print(f'  S = {s:.3f} mM   ->   v = {v:.2f} umol/min')

训练数据（实验员唯一能看到的东西）：
  S = 0.050 mM   ->   v = 1.52 umol/min
  S = 0.085 mM   ->   v = 1.47 umol/min
  S = 0.143 mM   ->   v = 3.62 umol/min
  S = 0.243 mM   ->   v = 5.10 umol/min
  S = 0.412 mM   ->   v = 4.91 umol/min
  S = 0.697 mM   ->   v = 6.84 umol/min
  S = 1.181 mM   ->   v = 9.04 umol/min
  S = 2.000 mM   ->   v = 9.81 umol/min


In [4]:
# ============================================================
# 第 2 步：GPR 拟合 + 95% 置信带
# ============================================================
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

# 核函数 = 常数项（信号幅度） x RBF（平滑性）
# length_scale 初值 0.3：允许曲线在 ~0.3 mM 尺度上弯曲
kernel = ConstantKernel(1.0) * RBF(length_scale=0.3)

# alpha = 噪声方差。噪声标准差是 5% x 12 = 0.6，方差 = 0.36
# （实战中它也会被自动优化，这里先告诉模型我们的测量精度）
gpr = GaussianProcessRegressor(kernel=kernel, alpha=0.36, n_restarts_optimizer=10,
                               random_state=42)

# sklearn 要求 X 是二维数组 (n_samples, n_features)
X_train = S_train.reshape(-1, 1)
gpr.fit(X_train, v_train)

print('优化后的核函数：', gpr.kernel_)
print('对数边际似然：', gpr.log_marginal_likelihood_value_)

# 在细网格上预测（注意：预测范围故意延伸到 4 mM，超出数据范围，看外推行为）
X_pred = S_fine.reshape(-1, 1)
v_mean, v_std = gpr.predict(X_pred, return_std=True)

# 95% 置信带：均值 ± 1.96 x 标准差（第 9 题的知识）
v_upper = v_mean + 1.96 * v_std
v_lower = v_mean - 1.96 * v_std

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(S_fine, v_lower, v_upper, color='#0279EE', alpha=0.2,
                label='95% confidence band')
ax.plot(S_fine, v_mean, color='#0279EE', lw=2, label='GPR mean prediction')
ax.plot(S_fine, v_true, 'k--', lw=1, label='true curve')
ax.scatter(S_train, v_train, s=60, color='#FF9400', zorder=3, label='measurements')
ax.axvline(2.0, color='gray', ls=':', lw=1)   # 数据范围右边界
ax.text(2.05, 2, 'extrapolation ->', color='gray', fontsize=9)
ax.set_xlabel('substrate concentration S (mM)')
ax.set_ylabel('reaction rate v (umol/min)')
ax.set_title('GPR fit with 95% confidence band')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('/mnt/results/project_step2_gpr_fit.png', dpi=150)
plt.show()

优化后的核函数： 8.23**2 * RBF(length_scale=1.72)
对数边际似然： -17.37191241455285


In [5]:
# ============================================================
# 第 3 步：对比实验 —— GPR vs 多项式拟合（外推行为）
# 多项式拟合：没有不确定度概念，外推区只能“硬猜”
# ============================================================
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.family'] = ['Liberation Sans', 'DejaVu Sans']

# 4 阶多项式拟合同样的 8 个点
poly_coef = np.polyfit(S_train, v_train, deg=4)
v_poly = np.polyval(poly_coef, S_fine)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

# 左图：GPR（同上一张图）
axes[0].fill_between(S_fine, v_lower, v_upper, color='#0279EE', alpha=0.2)
axes[0].plot(S_fine, v_mean, color='#0279EE', lw=2, label='GPR mean')
axes[0].plot(S_fine, v_true, 'k--', lw=1, label='true curve')
axes[0].scatter(S_train, v_train, s=50, color='#FF9400', zorder=3)
axes[0].axvline(2.0, color='gray', ls=':', lw=1)
axes[0].set_title('GPR: honest uncertainty')
axes[0].set_xlabel('S (mM)')
axes[0].set_ylabel('v (umol/min)')
axes[0].set_ylim(-8, 16)
axes[0].legend(loc='lower left')

# 右图：4 阶多项式
axes[1].plot(S_fine, v_poly, color='#FD9BED', lw=2, label='degree-4 polynomial')
axes[1].plot(S_fine, v_true, 'k--', lw=1, label='true curve')
axes[1].scatter(S_train, v_train, s=50, color='#FF9400', zorder=3)
axes[1].axvline(2.0, color='gray', ls=':', lw=1)
axes[1].set_title('Polynomial: confidently wrong')
axes[1].set_xlabel('S (mM)')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.savefig('/mnt/results/project_step3_gpr_vs_poly.png', dpi=150)
plt.show()

# 量化外推区的荒唐程度
for s_check in [3.0, 4.0]:
    print(f'S = {s_check} mM 处：')
    print(f'  真实值        ≈ {michaelis_menten(np.array([s_check]))[0]:7.2f}')
    print(f'  多项式预测    = {np.polyval(poly_coef, s_check):7.2f}  <- 毫无预警地偏离')
    i = np.argmin(np.abs(S_fine - s_check))
    print(f'  GPR 预测      = {v_mean[i]:7.2f} ± {1.96*v_std[i]:.2f}  <- 用宽度承认自己不确定')

S = 3.0 mM 处：
  真实值        ≈   10.59
  多项式预测    = -120.43  <- 毫无预警地偏离
  GPR 预测      =    6.90 ± 5.38  <- 用宽度承认自己不确定
S = 4.0 mM 处：
  真实值        ≈   10.91
  多项式预测    = -747.81  <- 毫无预警地偏离
  GPR 预测      =    3.57 ± 11.41  <- 用宽度承认自己不确定


In [6]:
# ============================================================
# 第 4 步：调参实验 —— 亲手看长度尺度和噪声如何塑造曲线
# 技巧：optimizer=None 固定超参数，看“如果模型这么想，会画出什么”
# 幅度常数固定为 10（与信号量级匹配），只改变我们要研究的那个参数
# ============================================================
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

configs = [
    ('length_scale = 0.1 (too small)', 0.1, 0.36),
    ('length_scale = 1.7 (auto-learned)', 1.7, 0.36),
    ('length_scale = 10 (too large)', 10.0, 0.36),
    ('noise alpha = 0 (overconfident)', 1.7, 1e-8),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)

for ax, (title, ls, alpha) in zip(axes.ravel(), configs):
    k = ConstantKernel(10.0) * RBF(length_scale=ls)   # 幅度固定为 10
    m = GaussianProcessRegressor(kernel=k, alpha=alpha, optimizer=None,
                                 random_state=42)
    m.fit(X_train, v_train)
    mu, sd = m.predict(X_pred, return_std=True)
    ax.fill_between(S_fine, mu-1.96*sd, mu+1.96*sd, color='#0279EE', alpha=0.2)
    ax.plot(S_fine, mu, color='#0279EE', lw=2)
    ax.plot(S_fine, v_true, 'k--', lw=1)
    ax.scatter(S_train, v_train, s=40, color='#FF9400', zorder=3)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(-5, 16)                                # 统一坐标，才能公平对比

fig.suptitle('Effect of length scale and noise on GPR behaviour', y=1.0)
plt.tight_layout()
plt.savefig('/mnt/results/project_step4_hyperparams.png', dpi=150)
plt.show()

In [7]:
import numpy as np

print('linspace(0.05, 2, 8)   线性等距：')
print(np.round(np.linspace(0.05, 2, 8), 3))
print()
print('geomspace(0.05, 2, 8)  几何等距（比值相同）：')
print(np.round(np.geomspace(0.05, 2, 8), 3))
print()
g = np.geomspace(0.05, 2, 8)
print('geomspace 相邻两点的比值：', np.round(g[1:]/g[:-1], 2))

linspace(0.05, 2, 8)   线性等距：
[0.05  0.329 0.607 0.886 1.164 1.443 1.721 2.   ]

geomspace(0.05, 2, 8)  几何等距（比值相同）：
[0.05  0.085 0.143 0.243 0.412 0.697 1.181 2.   ]

geomspace 相邻两点的比值： [1.69 1.69 1.69 1.69 1.69 1.69 1.69]


In [8]:
import numpy as np

# 创建生成器：seed=42 是“随机种子”
rng = np.random.default_rng(seed=42)

# 从这个生成器抽随机数
print('5 个标准正态样本：', np.round(rng.normal(0, 1, 5), 3))

# 关键性质：同样的种子 -> 完全相同的“随机”数序列
rng_a = np.random.default_rng(seed=42)
rng_b = np.random.default_rng(seed=42)
print('两个 seed=42 的生成器各抽 3 个数：')
print('  rng_a:', np.round(rng_a.normal(0, 1, 3), 3))
print('  rng_b:', np.round(rng_b.normal(0, 1, 3), 3))
print('  -> 完全一样，这就是“可复现”')

# 对比：旧式写法（项目里我第一版用的）
print('\n旧式全局写法 np.random.normal(0, 1, 3)：', np.round(np.random.normal(0, 1, 3), 3))

5 个标准正态样本： [ 0.305 -1.04   0.75   0.941 -1.951]
两个 seed=42 的生成器各抽 3 个数：
  rng_a: [ 0.305 -1.04   0.75 ]
  rng_b: [ 0.305 -1.04   0.75 ]
  -> 完全一样，这就是“可复现”

旧式全局写法 np.random.normal(0, 1, 3)： [ 0.945 -1.356  1.136]


In [9]:
# ============================================================
# 路线一：组合现成核（+ 和 * 就是加法和乘法）
# 数学性质：两个核相加/相乘仍是合法的核
# ============================================================
from sklearn.gaussian_process.kernels import RBF, ExpSineSquared, ConstantKernel, WhiteKernel

# 例：平滑趋势 + 周期波动（生物钟基因表达的典型选择）
combo_kernel = ConstantKernel(1.0) * RBF(length_scale=5.0) \
             + ConstantKernel(1.0) * ExpSineSquared(length_scale=1.0, periodicity=24.0)

print('组合核：', combo_kernel)
print()

# ============================================================
# 路线二：从零写一个自定义核类
# 需实现：__call__（完整协方差矩阵）、diag（对角线）、is_stationary
# ============================================================
from sklearn.gaussian_process.kernels import Kernel
from scipy.spatial.distance import cdist

class MyRBF(Kernel):
    """手写版 RBF 核：k(x, x') = exp(-0.5 * ||x-x'||^2 / length_scale^2)"""

    def __init__(self, length_scale=1.0):
        self.length_scale = length_scale

    def __call__(self, X, Y=None):
        X = np.atleast_2d(X)
        Y = X if Y is None else np.atleast_2d(Y)
        d = cdist(X, Y) / self.length_scale      # 归一化距离
        return np.exp(-0.5 * d**2)

    def diag(self, X):
        # 自己和自己的关联永远是 1
        return np.ones(X.shape[0])

    @property
    def is_stationary(self):
        # 只依赖距离差 x - x' 的核是“平稳核”
        return True

    def __repr__(self):
        return f'MyRBF(length_scale={self.length_scale})'

# 验证：手写核和 sklearn 官方 RBF 给出完全相同的协方差矩阵
X_test = np.linspace(0, 1, 5).reshape(-1, 1)
K_mine = MyRBF(length_scale=0.5)(X_test)
K_official = RBF(length_scale=0.5)(X_test)
print('手写核与官方 RBF 的最大差异：', np.abs(K_mine - K_official).max())

# 用自定义核拟合酶动力学数据（和第 2 步同样的流程）
gpr_custom = GaussianProcessRegressor(kernel=MyRBF(length_scale=1.7),
                                      alpha=0.36, optimizer=None, random_state=42)
gpr_custom.fit(X_train, v_train)
mu_c, sd_c = gpr_custom.predict(X_pred, return_std=True)
print(f'\n自定义核拟合成功：S=1.0 mM 处预测 {mu_c[np.argmin(np.abs(S_fine-1.0))]:.2f} umol/min')

组合核： 1**2 * RBF(length_scale=5) + 1**2 * ExpSineSquared(length_scale=1, periodicity=24)

手写核与官方 RBF 的最大差异： 0.0

自定义核拟合成功：S=1.0 mM 处预测 6.77 umol/min
